In [ ]:
#PARAMETERS (for papermill)

# Đường dẫn dữ liệu đã feature engineering từ notebook 03
INPUT_DATA_PATH = "data/processed/feature_engineered_data.parquet"

# Thư mục lưu kết quả
OUTPUT_DIR = "data/processed"

# Tên file output
BASELINE_RESULTS_FILENAME = "baseline_results.csv"
BASELINE_METRICS_FILENAME = "baseline_metrics.csv"

# Cột mục tiêu (cần dự báo)
TARGET_COLUMN = "Global_active_power"

# Tỷ lệ tập test (ví dụ: 20% dữ liệu cuối để test)
TEST_SIZE = 0.2

# Chu kỳ mùa vụ (24 giờ = 1 ngày)
SEASONAL_PERIOD = 24

# Bật/tắt các biểu đồ
PLOT_TRAIN_TEST_SPLIT = True
PLOT_BASELINE_FORECAST = True
PLOT_RESIDUALS = True

In [ ]:
# ==================== SETUP ====================
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Xác định project_root linh hoạt
cwd = os.getcwd()
if os.path.basename(cwd) == "notebooks":
    project_root = os.path.abspath("..")
else:
    project_root = cwd

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import thư viện tự viết
from energy_forecast_library import DataLoader, ForecastingModels, Evaluator, Utils

# Tạo thư mục output nếu chưa có
os.makedirs(os.path.join(project_root, OUTPUT_DIR), exist_ok=True)
os.makedirs(os.path.join(project_root, "outputs/figures"), exist_ok=True)

print(f"✅ Project root: {project_root}")
print(f"✅ Output dir: {OUTPUT_DIR}")
print("✅ Đã import thư viện thành công")

In [ ]:
# ==================== LOAD DATA ====================
data_path = os.path.join(project_root, INPUT_DATA_PATH)

if not os.path.exists(data_path):
    print(f"❌ Không tìm thấy file: {data_path}")
    print("Vui lòng chạy notebook 03 trước!")
else:
    df = DataLoader.load_processed_data(data_path)
    
    print(f"✅ Đã tải dữ liệu từ: {INPUT_DATA_PATH}")
    print(f"📊 Kích thước: {df.shape[0]:,} dòng, {df.shape[1]} cột")
    print(f"📅 Thời gian: {df.index.min()} → {df.index.max()}")
    print(f"📌 Cột mục tiêu: {TARGET_COLUMN}")
    
    df.head()

In [ ]:
# ==================== CHIA TRAIN/TEST ====================
print("📊 Chia dữ liệu train/test theo thời gian...")
print(f"📌 Tỷ lệ test: {TEST_SIZE*100:.0f}%")

# Chia dữ liệu
y_train, y_test, X_train, X_test = ForecastingModels.split_time_series(
    df, 
    target_col=TARGET_COLUMN, 
    test_size=TEST_SIZE
)

print(f"\n✅ Train: {len(y_train):,} samples ({y_train.index[0]} → {y_train.index[-1]})")
print(f"✅ Test: {len(y_test):,} samples ({y_test.index[0]} → {y_test.index[-1]})")

In [ ]:
# ==================== VẼ BIỂU ĐỒ TRAIN/TEST SPLIT ====================
if PLOT_TRAIN_TEST_SPLIT:
    fig, ax = plt.subplots(figsize=(15, 6))
    
    # Lấy mẫu để vẽ (tránh quá nhiều điểm)
    train_sample = y_train.iloc[-1000:]  # 1000 điểm cuối của train
    test_sample = y_test.iloc[:500]      # 500 điểm đầu của test
    
    ax.plot(train_sample.index, train_sample.values, 
            color='blue', linewidth=1, label='Train')
    ax.plot(test_sample.index, test_sample.values, 
            color='red', linewidth=1, label='Test')
    
    ax.axvline(x=y_test.index[0], color='black', linestyle='--', alpha=0.7, 
               label=f'Split point: {y_test.index[0].strftime("%Y-%m-%d")}')
    
    ax.set_xlabel('Thời gian')
    ax.set_ylabel(TARGET_COLUMN)
    ax.set_title('Phân chia Train/Test theo thời gian')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/train_test_split.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== MÔ HÌNH SEASONAL NAÏVE ====================
print("📈 Đang chạy mô hình Seasonal Naïve...")
print(f"📌 Chu kỳ mùa vụ: {SEASONAL_PERIOD} giờ (1 ngày)")

# Dự báo
y_pred_baseline = ForecastingModels.seasonal_naive(
    y_train, 
    y_test, 
    seasonal_period=SEASONAL_PERIOD
)

print("✅ Đã hoàn thành dự báo baseline")

# Tạo DataFrame kết quả
results_df = pd.DataFrame({
    'timestamp': y_test.index,
    'actual': y_test.values,
    'baseline_pred': y_pred_baseline.values
})

results_df.head(10)

In [ ]:
# ==================== ĐÁNH GIÁ MÔ HÌNH ====================
# Tính các metrics
mae = Evaluator.calculate_mae(y_test.values, y_pred_baseline.values)
rmse = Evaluator.calculate_rmse(y_test.values, y_pred_baseline.values)
smape = Evaluator.calculate_smape(y_test.values, y_pred_baseline.values)

print("="*50)
print("📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH BASELINE")
print("="*50)
print(f"✅ MAE  (Mean Absolute Error): {mae:.4f}")
print(f"✅ RMSE (Root Mean Square Error): {rmse:.4f}")
print(f"✅ SMAPE (Symmetric MAPE): {smape:.2f}%")
print("="*50)

# Lưu metrics
metrics_dict = {
    'model': 'Seasonal_Naive',
    'mae': mae,
    'rmse': rmse,
    'smape': smape,
    'seasonal_period': SEASONAL_PERIOD,
    'test_size': len(y_test),
    'train_size': len(y_train)
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df

In [ ]:
# ==================== VẼ BIỂU ĐỒ DỰ BÁO ====================
if PLOT_BASELINE_FORECAST:
    # Lấy mẫu 7 ngày đầu của test để dễ nhìn
    sample_days = 7
    sample_points = sample_days * 24  # 24 giờ mỗi ngày
    
    sample_results = results_df.iloc[:sample_points].copy()
    
    fig, ax = plt.subplots(figsize=(15, 6))
    
    ax.plot(sample_results['timestamp'], sample_results['actual'], 
            color='black', linewidth=1.5, label='Thực tế')
    ax.plot(sample_results['timestamp'], sample_results['baseline_pred'], 
            color='red', linestyle='--', linewidth=1.5, label='Dự báo (Seasonal Naïve)')
    
    ax.set_xlabel('Thời gian')
    ax.set_ylabel(TARGET_COLUMN)
    ax.set_title(f'Dự báo Baseline - {sample_days} ngày đầu tiên của tập test')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/baseline_forecast.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()
    
    # Vẽ zoom vào 48 giờ đầu
    fig, ax = plt.subplots(figsize=(15, 5))
    
    zoom_results = results_df.iloc[:48].copy()  # 48 giờ đầu
    
    ax.plot(zoom_results['timestamp'], zoom_results['actual'], 
            color='black', linewidth=2, marker='o', markersize=4, label='Thực tế')
    ax.plot(zoom_results['timestamp'], zoom_results['baseline_pred'], 
            color='red', linestyle='--', linewidth=2, marker='s', markersize=4, label='Dự báo')
    
    ax.set_xlabel('Thời gian')
    ax.set_ylabel(TARGET_COLUMN)
    ax.set_title('Dự báo Baseline - Zoom 48 giờ đầu')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/baseline_forecast_zoom.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== PHÂN TÍCH PHẦN DƯ ====================
residuals = y_test.values - y_pred_baseline.values

residual_analysis = Evaluator.residual_analysis(y_test.values, y_pred_baseline.values)

print("📊 Phân tích phần dư (Residuals):")
print(f"   - Mean: {residual_analysis['mean']:.4f}")
print(f"   - Std: {residual_analysis['std']:.4f}")
print(f"   - Skewness: {residual_analysis['skewness']:.4f}")
print(f"   - Kurtosis: {residual_analysis['kurtosis']:.4f}")
print(f"   - Min: {residual_analysis['min']:.4f}")
print(f"   - Max: {residual_analysis['max']:.4f}")

if abs(residual_analysis['mean']) < 0.1:
    print("✅ Residual mean gần 0 - tốt")
else:
    print("⚠️ Residual mean khác 0 - mô hình có thể bị bias")

if abs(residual_analysis['skewness']) < 0.5:
    print("✅ Residual gần đối xứng")
else:
    print("⚠️ Residual bị lệch")

In [ ]:
# ==================== VẼ BIỂU ĐỒ PHẦN DƯ ====================
if PLOT_RESIDUALS:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # 1. Residuals over time
    axes[0,0].plot(y_test.index, residuals, color='purple', linewidth=0.8)
    axes[0,0].axhline(y=0, color='red', linestyle='--', linewidth=1)
    axes[0,0].set_xlabel('Thời gian')
    axes[0,0].set_ylabel('Residuals')
    axes[0,0].set_title('Phần dư theo thời gian')
    axes[0,0].grid(True, alpha=0.3)
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. Histogram of residuals
    axes[0,1].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0,1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0,1].set_xlabel('Residuals')
    axes[0,1].set_ylabel('Tần suất')
    axes[0,1].set_title('Phân phối phần dư')
    axes[0,1].grid(True, alpha=0.3, axis='y')
    
    # 3. Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[1,0])
    axes[1,0].set_title('Q-Q Plot')
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Actual vs Predicted
    axes[1,1].scatter(y_test.values, y_pred_baseline.values, alpha=0.3, s=1, color='green')
    min_val = min(y_test.min(), y_pred_baseline.min())
    max_val = max(y_test.max(), y_pred_baseline.max())
    axes[1,1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    axes[1,1].set_xlabel('Giá trị thực tế')
    axes[1,1].set_ylabel('Giá trị dự báo')
    axes[1,1].set_title('Thực tế vs Dự báo')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/baseline_residuals.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== PHÂN TÍCH LỖI THEO THỜI GIAN ====================
# Tạo DataFrame với residuals và thông tin thời gian
residuals_df = pd.DataFrame({
    'timestamp': y_test.index,
    'actual': y_test.values,
    'predicted': y_pred_baseline.values,
    'residual': residuals,
    'absolute_error': np.abs(residuals),
    'percentage_error': np.abs(residuals) / y_test.values * 100
})

# Thêm các cột thời gian
residuals_df['hour'] = residuals_df['timestamp'].dt.hour
residuals_df['dayofweek'] = residuals_df['timestamp'].dt.dayofweek
residuals_df['month'] = residuals_df['timestamp'].dt.month
residuals_df['is_weekend'] = (residuals_df['dayofweek'] >= 5).astype(int)

print("📊 Phân tích lỗi theo giờ:")
error_by_hour = residuals_df.groupby('hour')['absolute_error'].mean()
for hour in [0, 6, 12, 18, 23]:
    print(f"   - Giờ {hour}: {error_by_hour[hour]:.4f}")

print("\n📊 Phân tích lỗi theo thứ:")
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
error_by_dow = residuals_df.groupby('dayofweek')['absolute_error'].mean()
for dow in range(7):
    print(f"   - {dow_labels[dow]}: {error_by_dow[dow]:.4f}")

print(f"\n📊 Lỗi trung bình:")
print(f"   - Ngày thường: {residuals_df[~residuals_df['is_weekend'].astype(bool)]['absolute_error'].mean():.4f}")
print(f"   - Cuối tuần: {residuals_df[residuals_df['is_weekend'].astype(bool)]['absolute_error'].mean():.4f}")

In [ ]:
# ==================== VẼ BIỂU ĐỒ LỖI THEO THỜI GIAN ====================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lỗi theo giờ
error_by_hour.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_xlabel('Giờ trong ngày')
axes[0].set_ylabel('MAE')
axes[0].set_title('Lỗi trung bình theo giờ')
axes[0].grid(True, alpha=0.3, axis='y')

# Lỗi theo thứ
error_by_dow.index = dow_labels
error_by_dow.plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Thứ')
axes[1].set_ylabel('MAE')
axes[1].set_title('Lỗi trung bình theo thứ')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs/figures/baseline_error_by_time.png'), 
            dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== SO SÁNH VỚI CÁC PHƯƠNG PHÁP ĐƠN GIẢN ====================
# Thử thêm 2 phương pháp đơn giản khác để so sánh

# 1. Persistence Model (dự báo bằng giá trị cuối cùng của train)
persistence_pred = [y_train.iloc[-1]] * len(y_test)

# 2. Mean Model (dự báo bằng trung bình của train)
mean_pred = [y_train.mean()] * len(y_test)

# Tính metrics
metrics_comparison = []

# Baseline (Seasonal Naive)
metrics_comparison.append({
    'model': 'Seasonal Naive (24h)',
    'mae': Evaluator.calculate_mae(y_test.values, y_pred_baseline.values),
    'rmse': Evaluator.calculate_rmse(y_test.values, y_pred_baseline.values),
    'smape': Evaluator.calculate_smape(y_test.values, y_pred_baseline.values)
})

# Persistence
metrics_comparison.append({
    'model': 'Persistence (last value)',
    'mae': Evaluator.calculate_mae(y_test.values, persistence_pred),
    'rmse': Evaluator.calculate_rmse(y_test.values, persistence_pred),
    'smape': Evaluator.calculate_smape(y_test.values, persistence_pred)
})

# Mean
metrics_comparison.append({
    'model': 'Mean',
    'mae': Evaluator.calculate_mae(y_test.values, mean_pred),
    'rmse': Evaluator.calculate_rmse(y_test.values, mean_pred),
    'smape': Evaluator.calculate_smape(y_test.values, mean_pred)
})

comparison_df = pd.DataFrame(metrics_comparison)
print("📊 SO SÁNH CÁC MÔ HÌNH ĐƠN GIẢN:")
comparison_df

In [ ]:
# ==================== NHẬN XÉT VÀ KẾT LUẬN ====================
print("="*60)
print("📝 NHẬN XÉT MÔ HÌNH BASELINE")
print("="*60)

print("""
✅ ƯU ĐIỂM:
   - Đơn giản, dễ hiểu
   - Chạy nhanh, không cần huấn luyện
   - Nắm bắt được tính mùa vụ cơ bản (theo ngày)
   - Làm baseline tốt để so sánh với các mô hình phức tạp hơn

⚠️ HẠN CHẾ:
   - Chỉ dùng được khi có tính mùa vụ rõ ràng
   - Không học được xu hướng mới
   - Nhạy cảm với nhiễu
   - Lỗi có thể tích lũy theo thời gian
""")

print(f"\n📊 CHỈ SỐ BASELINE:")
print(f"   - MAE: {mae:.4f}")
print(f"   - RMSE: {rmse:.4f}")
print(f"   - SMAPE: {smape:.2f}%")

print(f"\n🔍 PHÂN TÍCH LỖI:")
best_hour = error_by_hour.idxmin()
worst_hour = error_by_hour.idxmax()
print(f"   - Giờ dự báo tốt nhất: {best_hour}h (MAE = {error_by_hour[best_hour]:.4f})")
print(f"   - Giờ dự báo tệ nhất: {worst_hour}h (MAE = {error_by_hour[worst_hour]:.4f})")

best_dow = error_by_dow.idxmin()  # Vẫn là string
worst_dow = error_by_dow.idxmax() # Vẫn là string
print(f"   - Thứ dự báo tốt nhất: {best_dow} (MAE = {error_by_dow[best_dow]:.4f})")
print(f"   - Thứ dự báo tệ nhất: {worst_dow} (MAE = {error_by_dow[worst_dow]:.4f})")

print("\n👉 Bước tiếp theo: Sử dụng các mô hình phức tạp hơn (ARIMA, ETS)")
print("   để cải thiện độ chính xác dự báo.")

In [ ]:
# ==================== LƯU KẾT QUẢ ====================
# Lưu kết quả dự báo
results_path = os.path.join(project_root, OUTPUT_DIR, BASELINE_RESULTS_FILENAME)
results_df.to_csv(results_path, index=False)

# Lưu metrics
metrics_path = os.path.join(project_root, OUTPUT_DIR, BASELINE_METRICS_FILENAME)
metrics_df.to_csv(metrics_path, index=False)

# Lưu so sánh các methods
comparison_path = os.path.join(project_root, OUTPUT_DIR, 'simple_methods_comparison.csv')
comparison_df.to_csv(comparison_path, index=False)

# Lưu residuals analysis
residuals_df.to_csv(os.path.join(project_root, OUTPUT_DIR, 'baseline_residuals.csv'), index=False)

print("\n✅ Đã lưu kết quả thành công:")
print(f"   - Dự báo: {BASELINE_RESULTS_FILENAME}")
print(f"   - Metrics: {BASELINE_METRICS_FILENAME}")
print(f"   - So sánh methods: simple_methods_comparison.csv")
print(f"   - Residuals: baseline_residuals.csv")
print(f"\n✅ Sẵn sàng cho bước tiếp theo: 05_arima_forecasting.ipynb")

In [ ]:
# ==================== KIỂM TRA NHANH ====================
if os.path.exists(results_path):
    test_results = pd.read_csv(results_path)
    print(f"✅ Kiểm tra: Đã đọc lại được file {BASELINE_RESULTS_FILENAME}")
    print(f"   - Số dòng: {test_results.shape[0]:,}")
    print(f"   - Các cột: {list(test_results.columns)}")
    print(f"   - 5 dòng đầu:")
    print(test_results.head())